# Spartan Search

Spartan (`spartan://`) 用の小さなクローラー + SQLite全文検索エンジンです。

- TCP/300 で Spartan を直接取得
- Gemtext の `=>` リンクを巡回
- `=:` 入力リンクとクエリ付きURLは自動送信しない
- `text/gemini` / `text/plain` を SQLite FTS5 に索引
- FTS5 `trigram` が使える環境では日本語検索にも対応

**注意:** Smolnet のサーバーは個人運営・低リソースなものが多いため、`DELAY` は小さくしすぎないでください。

In [ ]:
from __future__ import annotations

import posixpath
import socket
import sqlite3
import time
from collections import deque
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import quote, urlsplit, urlunsplit

DEFAULT_PORT = 300
DB_PATH = "spartan.db"
TEXT_MIMES = {"text/gemini", "text/plain"}

@dataclass(slots=True)
class Response:
    status: int
    meta: str
    body: bytes

## Database

In [ ]:
def connect_db(path: str = DB_PATH) -> sqlite3.Connection:
    db = sqlite3.connect(path)
    db.row_factory = sqlite3.Row
    db.executescript("""
        PRAGMA journal_mode=WAL;
        CREATE TABLE IF NOT EXISTS pages (
            url TEXT PRIMARY KEY,
            title TEXT NOT NULL DEFAULT '',
            mime TEXT NOT NULL DEFAULT '',
            body TEXT NOT NULL DEFAULT '',
            status INTEGER NOT NULL DEFAULT 0,
            error TEXT NOT NULL DEFAULT '',
            fetched_at TEXT NOT NULL
        );
        CREATE TABLE IF NOT EXISTS links (
            source TEXT NOT NULL,
            target TEXT NOT NULL,
            PRIMARY KEY (source, target)
        );
        CREATE INDEX IF NOT EXISTS links_target_idx ON links(target);
    """)
    try:
        db.execute("CREATE VIRTUAL TABLE IF NOT EXISTS pages_fts USING fts5(url UNINDEXED, title, body, tokenize='trigram')")
    except sqlite3.OperationalError:
        db.execute("CREATE VIRTUAL TABLE IF NOT EXISTS pages_fts USING fts5(url UNINDEXED, title, body, tokenize='unicode61')")
    return db

## Spartan client and URL handling

In [ ]:
def normalize_url(url: str) -> str | None:
    parts = urlsplit(url.strip())
    if parts.scheme.lower() != "spartan" or not parts.hostname or parts.query:
        return None
    try:
        host = parts.hostname.encode("idna").decode("ascii").lower()
    except UnicodeError:
        return None
    path = quote(parts.path or "/", safe="/%:@!$&'()*+,;=-._~")
    path = "/" + posixpath.normpath(path).lstrip("/")
    if (parts.path or "/").endswith("/") and not path.endswith("/"):
        path += "/"
    port = parts.port
    netloc = host if port in (None, DEFAULT_PORT) else f"{host}:{port}"
    return urlunsplit(("spartan", netloc, path, "", ""))

def resolve_url(base: str, target: str) -> str | None:
    target = target.strip()
    if not target or target.startswith(("#", "=:", "mailto:")):
        return None
    parsed = urlsplit(target)
    if parsed.scheme:
        return normalize_url(target)
    base_parts = urlsplit(base)
    if target.startswith("//"):
        return normalize_url(f"spartan:{target}")
    if target.startswith("/"):
        path = target
    else:
        directory = base_parts.path if base_parts.path.endswith("/") else posixpath.dirname(base_parts.path) + "/"
        path = posixpath.join(directory, target)
    return normalize_url(urlunsplit(("spartan", base_parts.netloc, path, "", "")))

def fetch_spartan(url: str, timeout: float = 10.0, max_bytes: int = 2_000_000) -> Response:
    parts = urlsplit(url)
    if not parts.hostname:
        raise ValueError("URL has no host")
    host = parts.hostname.encode("idna").decode("ascii")
    port = parts.port or DEFAULT_PORT
    request = f"{host} {parts.path or '/'} 0\r\n".encode("ascii")
    with socket.create_connection((host, port), timeout=timeout) as sock:
        sock.settimeout(timeout)
        sock.sendall(request)
        stream = sock.makefile("rb")
        line = stream.readline(4097)
        if not line or len(line) > 4096 or not line.endswith(b"\n"):
            raise OSError("invalid Spartan response")
        line = line.rstrip(b"\r\n")
        if not line or line[:1] not in b"2345":
            raise OSError(f"invalid Spartan status: {line[:32]!r}")
        status = int(chr(line[0]))
        meta = line[1:].lstrip(b" ").decode("utf-8", "replace")
        if status != 2:
            return Response(status, meta, b"")
        body = stream.read(max_bytes + 1)
        if len(body) > max_bytes:
            raise OSError(f"response exceeds {max_bytes} bytes")
        return Response(status, meta, body)

## Gemtext parsing and indexing

In [ ]:
def parse_text(body: bytes, meta: str) -> tuple[str, str]:
    mime = meta.split(";", 1)[0].strip().lower() or "application/octet-stream"
    charset = "utf-8"
    for item in meta.split(";")[1:]:
        key, sep, value = item.strip().partition("=")
        if sep and key.lower() == "charset":
            charset = value.strip().strip('\"')
    try:
        return mime, body.decode(charset, "replace")
    except LookupError:
        return mime, body.decode("utf-8", "replace")

def parse_gemtext(text: str, base_url: str) -> tuple[str, str, list[str]]:
    title = ""
    searchable = []
    links = []
    in_pre = False
    for raw in text.splitlines():
        if raw.startswith("```"):
            in_pre = not in_pre
            continue
        if in_pre:
            searchable.append(raw)
            continue
        if raw.startswith("=:"):
            continue
        if raw.startswith("=>"):
            rest = raw[2:].strip()
            if not rest:
                continue
            target, _, label = rest.partition(" ")
            resolved = resolve_url(base_url, target)
            if resolved:
                links.append(resolved)
            if label.strip():
                searchable.append(label.strip())
            continue
        line = raw
        if raw.startswith("#"):
            line = raw.lstrip("#").strip()
            if not title and line:
                title = line
        elif raw.startswith("*"):
            line = raw[1:].strip()
        elif raw.startswith(">"):
            line = raw[1:].strip()
        if line.strip():
            searchable.append(line.strip())
    return title, "\n".join(searchable), links

def save_page(db, url, title, mime, body, status, error, links):
    db.execute("""
        INSERT INTO pages(url, title, mime, body, status, error, fetched_at)
        VALUES(?, ?, ?, ?, ?, ?, ?)
        ON CONFLICT(url) DO UPDATE SET
            title=excluded.title, mime=excluded.mime, body=excluded.body,
            status=excluded.status, error=excluded.error, fetched_at=excluded.fetched_at
    """, (url, title, mime, body, status, error, datetime.now(timezone.utc).isoformat()))
    db.execute("DELETE FROM pages_fts WHERE url = ?", (url,))
    if status == 2 and body:
        db.execute("INSERT INTO pages_fts(url, title, body) VALUES(?, ?, ?)", (url, title, body))
    db.execute("DELETE FROM links WHERE source = ?", (url,))
    db.executemany("INSERT OR IGNORE INTO links(source, target) VALUES(?, ?)", ((url, target) for target in links))
    db.commit()

## Crawler

In [ ]:
def crawl(seeds: list[str], max_pages: int = 500, delay: float = 1.0, timeout: float = 10.0, max_bytes: int = 2_000_000, db_path: str = DB_PATH):
    db = connect_db(db_path)
    queue = deque(filter(None, (normalize_url(seed) for seed in seeds)))
    queued = set(queue)
    last_fetch = {}
    crawled = 0
    while queue and crawled < max_pages:
        url = queue.popleft()
        parts = urlsplit(url)
        host_key = parts.netloc.lower()
        wait = delay - (time.monotonic() - last_fetch.get(host_key, 0.0))
        if wait > 0:
            time.sleep(wait)
        print(f"[{crawled + 1}/{max_pages}] {url}")
        links = []
        try:
            response = fetch_spartan(url, timeout=timeout, max_bytes=max_bytes)
            last_fetch[host_key] = time.monotonic()
            if response.status == 3:
                redirect = resolve_url(url, response.meta)
                save_page(db, url, "", "", "", 3, f"redirect: {response.meta}", [])
                if redirect and urlsplit(redirect).netloc == parts.netloc and redirect not in queued:
                    queue.append(redirect)
                    queued.add(redirect)
            elif response.status == 2:
                mime, text = parse_text(response.body, response.meta)
                if mime == "text/gemini":
                    title, searchable, links = parse_gemtext(text, url)
                    save_page(db, url, title, mime, searchable, 2, "", links)
                elif mime == "text/plain":
                    title = next((line.strip() for line in text.splitlines() if line.strip()), "")[:200]
                    save_page(db, url, title, mime, text, 2, "", [])
                else:
                    save_page(db, url, "", mime, "", 2, "", [])
            else:
                save_page(db, url, "", "", "", response.status, response.meta, [])
        except Exception as exc:
            last_fetch[host_key] = time.monotonic()
            save_page(db, url, "", "", "", 0, str(exc), [])
            print("  !", exc)
        for link in links:
            if link not in queued:
                queue.append(link)
                queued.add(link)
        crawled += 1
    db.close()
    print(f"done: crawled={crawled}, queued={len(queue)}, db={db_path}")

## Crawl settings

`SEEDS` は分かっている Spartan サーバーを複数入れてOKです。

In [ ]:
SEEDS = [
    "spartan://spartan.mozz.us/",
]
MAX_PAGES = 500
DELAY = 1.0
TIMEOUT = 10.0
MAX_BYTES = 2_000_000

# 実行すると実際にネットワークへアクセスします。
# crawl(SEEDS, MAX_PAGES, DELAY, TIMEOUT, MAX_BYTES)

`crawl(...)` の先頭の `#` を外すか、次のセルで実行してください。

In [ ]:
# crawl(SEEDS, max_pages=MAX_PAGES, delay=DELAY, timeout=TIMEOUT, max_bytes=MAX_BYTES)

## Search

In [ ]:
def search(query: str, limit: int = 20, db_path: str = DB_PATH):
    query = query.strip()
    if not query:
        return []
    with connect_db(db_path) as db:
        if len(query) < 3:
            pattern = f"%{query}%"
            return db.execute("SELECT url, title, body, 0.0 AS score FROM pages WHERE status=2 AND (title LIKE ? OR body LIKE ?) LIMIT ?", (pattern, pattern, limit)).fetchall()
        try:
            match = " AND ".join(f'\"{term.replace(chr(34), chr(34) * 2)}\"' for term in query.split())
            return db.execute("SELECT url, title, body, bm25(pages_fts, 0.0, 4.0, 1.0) AS score FROM pages_fts WHERE pages_fts MATCH ? ORDER BY score LIMIT ?", (match, limit)).fetchall()
        except sqlite3.OperationalError:
            pattern = f"%{query}%"
            return db.execute("SELECT url, title, body, 0.0 AS score FROM pages WHERE status=2 AND (title LIKE ? OR body LIKE ?) LIMIT ?", (pattern, pattern, limit)).fetchall()

def snippet(body: str, query: str, width: int = 180) -> str:
    flat = " ".join(body.split())
    pos = flat.casefold().find(query.casefold())
    if pos < 0:
        return flat[:width] + ("…" if len(flat) > width else "")
    start = max(0, pos - width // 3)
    end = min(len(flat), start + width)
    return ("…" if start else "") + flat[start:end] + ("…" if end < len(flat) else "")

def print_results(query: str, limit: int = 20):
    rows = search(query, limit)
    for i, row in enumerate(rows, 1):
        print(f"{i}. {row['title'] or '(untitled)'}")
        print("  ", row["url"])
        print("  ", snippet(row["body"], query))
        print()
    print(f"{len(rows)} result(s)")

In [ ]:
QUERY = "Spartan"
print_results(QUERY)

## Stats

In [ ]:
with connect_db() as db:
    total = db.execute("SELECT COUNT(*) FROM pages").fetchone()[0]
    indexed = db.execute("SELECT COUNT(*) FROM pages WHERE status=2 AND body <> ''").fetchone()[0]
    errors = db.execute("SELECT COUNT(*) FROM pages WHERE status=0").fetchone()[0]
    hosts = db.execute("SELECT COUNT(DISTINCT substr(url, 11, instr(substr(url, 11), '/') - 1)) FROM pages").fetchone()[0]
print({"pages": total, "indexed": indexed, "errors": errors, "hosts": hosts})

## Colab: DBを保存

Colabのランタイムを消す前に必要なら `spartan.db` をダウンロードしてください。

In [ ]:
# Colab only:
# from google.colab import files
# files.download(DB_PATH)